# 07 — Styling: Colours and Line Widths

Every renderer in eucare reads three attribute keys off graph elements:

- `obj['color_key']` — a colour. Either an `(r, g, b)` / `(r, g, b, a)` tuple in `[0, 1]`, a hex string, or any hashable value. Non-colour hashables are mapped to distinct palette slots (matplotlib `tab10` by default, or evenly-spaced `hsv` samples once you exceed 10 classes), sorted by class frequency.
- `edge['line_width']` and `vertex['line_width']` — per-element stroke widths in figure units.
- `face['color_key']` — face fill colour (only applied when `render_faces=True`).

This notebook shows four ways to set those colours:

1. *Ephemerally at render time* via `face_color_by=` / `edge_color_by=` / `vertex_color_by=` on `G.show()` — the recommended path; no mutation.
2. *Mutating, via classifiers* in [`eucare.classifiers`](../reference/eucare/classifiers.md) — useful when you want the assignment baked into the graph (e.g. before serialisation or Conway-operator propagation).
3. *By hand* on faces and edges chosen by predicate.
4. *Inherited* via the `pre_conway` attribute that Conway operators leave behind on every new vertex / face / edge.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    shrink_rotate,
    rendering,
)
from eucare.rendering import multi_show


## Defaults: the crease-pattern preset

`rendering.CREASE_PATTERN_PRESET` is the dict we've been using throughout the series; alongside it sit three colour constants matching common origami conventions.

In [ ]:
print(rendering.CREASE_PATTERN_PRESET)
print('mountain:', rendering.MOUNTAIN_COLOR,
      'valley:', rendering.VALLEY_COLOR,
      'flat:', rendering.FLAT_COLOR)


## Render-time auto-colouring (no graph mutation)

The simplest path: ask `G.show()` to colour at render time without ever writing to `face.attributes`. Pass a preset name, a `Classifier`, or any callable returning a hashable.

Presets:

- **faces:** `"congruency"`, `"order"` (= side count)
- **edges:** `"length"`, `"orientation"` (angle mod π)
- **vertices:** `"order"` (= degree)

The default colormap is `tab10` (great for ≤ 10 classes); past that, the palette switches to evenly-spaced `hsv` so every class still gets a distinct hue. Override per-call with `face_cmap=`, `edge_cmap=`, `vertex_cmap=` — accepting any matplotlib colormap name, a `Colormap` instance, or a list of colours. Qualitative cmaps (`tab10`, `Set2`, ...) are used directly; continuous cmaps (`viridis`, `plasma`, ...) are sampled across their full `[0, 1]` range so categorical inputs span the whole gradient. Project-wide defaults live in `colorization.DEFAULT_FACE_CMAP` etc.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()

multi_show(
    [G, G, G, G],
    titles=[
        'by face order (tab10)',
        'by face congruence (tab10)',
        'by face congruence (viridis)',
        'by edge orientation',
    ],
    render_faces=True, face_inset=0.05, render_vertices=False,
    per_subplot_kwargs=[
        dict(face_color_by='order'),
        dict(face_color_by='congruency'),
        dict(face_color_by='congruency', face_cmap='viridis'),
        dict(edge_color_by='orientation', render_faces=False),
    ],
)
# Note: G is unchanged — no 'color_key' was written to any face or edge.
assert not any('color_key' in f.attributes for f in G.faces)

## Mutating colouring 1: by number of corners

When you do want the assignment baked into the graph — for serialisation, propagation through Conway operators (see the `pre_conway` section below), or because you'll render the same coloured graph many times — use `colorization.colorize()`. It writes a hashable class id to `face['color_key']`, and the renderer's palette resolution maps that id to a colour just like the `face_color_by=` path above.

We use `t_4_6_12` again. `LenClassifier` plus a one-line `lambda` extracts the corner count and `colorize` writes the result into `face['color_key']`. (The render-time equivalent is `G.show(face_color_by='order')`.)

In [ ]:
from eucare.classifiers import LenClassifier, PreMapClassifier

G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()
G_corners = G.copy()
corner_classifier = PreMapClassifier(LenClassifier(),
                                     lambda f: list(f.halfedge_iter()))
colorization.colorize(G_corners, corner_classifier)
multi_show([G, G_corners],
           titles=['plain', 'colored by corner count'],
           render_faces=True, face_inset=0.05, render_vertices=False)


## Mutating colouring 2: by congruence

Same pattern, but with the polygon-congruence classifier. (The render-time equivalent is `G.show(face_color_by='congruency')`.)

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()
hexagons_and_dodecagons = [f for f in G.faces if f.order() == 6 or f.order() == 12]
G = conway.kis_graph()(G, faces=hexagons_and_dodecagons, delete_on_border=False)
G.recompute_lengths_and_angles()

G_len  = G.copy(); colorization.colorize(G_len, corner_classifier)
G_cong = G.copy(); colorization.congruency_colorize(G_cong)

n_len  = len({f['color_key'] for f in G_len.faces  if 'color_key' in f.attributes})
n_cong = len({f['color_key'] for f in G_cong.faces if 'color_key' in f.attributes})
print(f'classes — by len: {n_len}, by congruence: {n_cong}')

multi_show([G_len, G_cong],
           titles=[f'by corner count ({n_len} classes)',
                   f'by congruence ({n_cong} classes)'],
           render_faces=True, face_inset=0.05, render_edges=False, render_vertices=False, face_cmap='tab10')


## Hand-assigned face colours

For one-off highlights, just write to `face['color_key']` directly. Pick faces with any predicate; below we paint everything within a small radius of the origin orange.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G.recompute_lengths_and_angles()
for f in G.faces:
    if np.linalg.norm(f.midpoint()) < 1.5:
        f['color_key'] = (1.0, 0.6, 0.0, 0.9)  # opaque orange
G.show(render_faces=True, face_inset=0.05, render_vertices=False)


## Hand-assigned edge colours and widths

Per-edge `'color_key'` and `'line_width'` work the same way. Below we make the boundary halfedges of one chosen face thick and red. Note that we set both directions of every edge.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
central = G.central_face()
for h in central.halfedge_iter():
    h['color_key'] = h.rev['color_key'] = (0.85, 0.10, 0.10, 1.0)
    h['line_width'] = h.rev['line_width'] = 0.2
G.show(face_inset=0.05, render_vertices=False)


## Inheriting colours via `pre_conway`

Every Conway operator stores some back-pointers on its outputs: some of the new vertex / face / halfedges in the result graph carry `obj['pre_conway']` → the corresponding element of the *input* graph. We can use that to propagate any per-face attribute through an operator, for example for coloring

In [ ]:
from eucare.half import Face

G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=1)
central_face = G.central_face()
central_face['color_key'] = (1.0, 0.5, 0.1, 0.9)

for v in central_face.vertex_iter():
    v['color_key'] = np.random.rand(3)

D = conway.chamfer_graph()(G.copy(), delete_on_border=False)
D.recompute_lengths_and_angles()
for obj in D.vertices.union(D.faces).union(D.halfedges):
    src = obj.attributes.get('pre_conway')
    if isinstance(src, Face) and 'color_key' in src.attributes:
        obj['color_key'] = src['color_key']

multi_show([G, D],
           titles=['original', 'dual (colors inherited from original)'],
           render_faces=True, render_vertices=True, face_inset=0,
           vertex_radius=0.1)
